## ejecutado en Colab GPU por el coste de ~120 entrenamientos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
RUTA = "/content/drive/MyDrive/wf_lstm_ensemble.csv"

In [ ]:
import numpy as np, pandas as pd, random
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

print("GPU:", tf.config.list_physical_devices("GPU"))   # debe mostrar un PhysicalDevice, no []

RUTA = "/content/drive/MyDrive/wf_lstm_ensemble.csv"

d = np.load("/content/drive/MyDrive/lstm_tensores.npz")
X, y = d["X"], d["y"]
ts = pd.to_datetime(d["ts"], utc=True)
meses = ts.to_period("M")
L, F = X.shape[1], X.shape[2]

SEEDS  = [0, 1, 2]     # ensemble de 3
INICIO = 12            # mismos folds que XGBoost: empieza a predecir en 2024-01

def construir():
    m = Sequential([Input(shape=(L, F)), LSTM(32), Dense(1)])
    m.compile(optimizer="adam", loss="mae")
    return m

def mae(a, b): return float(np.mean(np.abs(a - b)))

meses_unicos = np.sort(meses.unique())
filas = []
for mes in meses_unicos[INICIO:]:
    tr, te = (meses < mes), (meses == mes)
    if te.sum() == 0:
        continue
    Xtr, ytr, Xte, yte = X[tr], y[tr], X[te], y[te]

    # escalado POR FOLD (fit solo en train) — cero leakage
    sx = StandardScaler().fit(Xtr.reshape(-1, F))
    Xtr_s = sx.transform(Xtr.reshape(-1, F)).reshape(Xtr.shape)
    Xte_s = sx.transform(Xte.reshape(-1, F)).reshape(Xte.shape)
    sy = StandardScaler().fit(ytr.reshape(-1, 1))
    ytr_s = sy.transform(ytr.reshape(-1, 1)).ravel()

    # ensemble de semillas
    preds = []
    for s in SEEDS:
        np.random.seed(s); random.seed(s); tf.random.set_seed(s)
        modelo = construir()
        es = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
        modelo.fit(Xtr_s, ytr_s, validation_split=0.1, epochs=50,
                   batch_size=64, callbacks=[es], verbose=0)
        p = sy.inverse_transform(modelo.predict(Xte_s, verbose=0).reshape(-1, 1)).ravel()
        preds.append(p)
    pred_ens = np.mean(preds, axis=0)

    filas.append({"mes": str(mes), "MAE": mae(yte, pred_ens), "n": int(te.sum())})
    print(filas[-1])
    pd.DataFrame(filas).to_csv(RUTA, index=False)   # <-- guardado incremental (fold a fold)

res = pd.DataFrame(filas)
res.to_csv(RUTA, index=False)
print(f"\n=== LSTM ensemble walk-forward ===")
print(f"MAE {res['MAE'].mean():.2f} ± {res['MAE'].std():.2f}  ({len(res)} meses)")